# 02 – Mô hình baseline

**Baseline** là mô hình đơn giản đầu tiên, dùng làm *mốc để so sánh*. Mọi cải tiến ở notebook 03 đều phải thắng được baseline thì mới có ý nghĩa.

**Baseline ở đây:** MobileNetV2 (mạng đã học sẵn từ bộ ảnh ImageNet) được **đóng băng**, chỉ huấn luyện phần đầu phân loại nhỏ ở trên cùng để chia ra hai nhãn `fresh` / `rotten`. Cách này gọi là *transfer learning*: tận dụng kiến thức nhận dạng ảnh có sẵn, nên cần ít dữ liệu và train nhanh.

**Lưu ý:** notebook này **không ghi đè** `models/best_model.h5`. Biểu đồ và báo cáo được lưu vào `reports/` với tiền tố `baseline_`.

## 1. Chuẩn bị và xem cấu hình

`DEFAULT_CONFIG` là cấu hình baseline. Ý nghĩa từng tham số:

| Tham số | Ý nghĩa |
|---|---|
| `arch` | Kiến trúc: `mobilenetv2` (có pretrained) hoặc `cnn` (tự xây, không pretrained) |
| `lr` | Learning rate: bước học mỗi lần cập nhật. Lớn thì học nhanh nhưng dễ dao động, nhỏ thì ổn định nhưng chậm |
| `dropout` | Tỉ lệ nơ-ron bị tắt ngẫu nhiên khi train (0.3 = 30%) để chống học vẹt (overfitting) |
| `dense_units` | Số nơ-ron của lớp ẩn ở đầu phân loại |
| `aug` | Mức tăng cường dữ liệu: `normal` hoặc `strong` (xoay, dịch, zoom, đổi độ sáng...) |
| `finetune_layers` | Số lớp cuối của MobileNetV2 được mở khóa để train tiếp. 0 nghĩa là đóng băng hoàn toàn |
| `finetune_lr` | Learning rate (rất nhỏ) dùng ở giai đoạn fine-tune |

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # để import được package src

from IPython.display import Image

from src.data_loader import REPORTS_DIR, load_splits
from src.evaluate import evaluate_model, plot_history
from src.train import DEFAULT_CONFIG, fit_model, make_generators

cfg = DEFAULT_CONFIG
print(cfg)

## 2. Chia dữ liệu và tạo generator

**`load_splits()`** chia dữ liệu thành 3 tập với 3 vai trò khác nhau:

| Tập | Lấy từ | Dùng để |
|---|---|---|
| **train** | 80% của `data/raw/train` | Model *học* trên tập này |
| **val** (validation) | 20% còn lại của `data/raw/train` | *Theo dõi* trong lúc train và *chọn* cấu hình tốt nhất |
| **test** | `data/raw/test` | Chấm điểm *cuối cùng*, model chưa từng thấy |

Việc chia được chia đều theo từng loại trái cây (`stratify`), và lưu vào `data/processed/*.csv` để mọi lần chạy dùng đúng cùng một cách chia.

**`make_generators()`** tạo bộ nạp ảnh theo từng lô (batch 32 ảnh) cho model:
- Tập **train** có *augmentation*: mỗi lần nạp ảnh bị xoay, dịch, zoom, đổi sáng ngẫu nhiên, giúp model quen với nhiều biến thể.
- Tập **val** và **test** *không* augmentation (ảnh giữ nguyên) và `shuffle=False` để thứ tự ảnh cố định, đảm bảo đánh giá chính xác.
- Mọi tập đều resize về 224×224 và chuẩn hóa pixel về khoảng [-1, 1] bằng đúng hàm của MobileNetV2. API cũng dùng đúng cách xử lý này.

In [ ]:
train_df, val_df, test_df = load_splits()  # lần đầu sẽ tạo data/interim và data/processed
train_gen, val_gen, test_gen = make_generators(train_df, val_df, test_df, cfg['aug'])

## 3. Huấn luyện

`fit_model()` xây model theo cấu hình rồi huấn luyện tối đa 15 epoch (1 epoch = model nhìn hết tập train một lượt). Hai cơ chế tự động giúp quá trình tốt hơn:

- **EarlyStopping:** nếu `val_loss` không giảm sau 4 epoch liên tiếp thì dừng sớm và *khôi phục trọng số tốt nhất*. Tránh train quá lâu gây overfitting.
- **ReduceLROnPlateau:** nếu `val_loss` đứng yên 2 epoch thì giảm learning rate xuống còn 30%, để model tinh chỉnh kỹ hơn.

Hàm trả về 3 thứ: `model` (đã train), `history` (số liệu từng epoch) và `val` (điểm cuối trên tập validation). Trong `val`, `val_acc` là độ chính xác (càng cao càng tốt), `val_loss` là độ sai lệch (càng thấp càng tốt).

Muốn chạy thử nhanh, đổi `epochs=15` thành `epochs=3`.

In [ ]:
model, history, val = fit_model(cfg, train_gen, val_gen, epochs=15)
print('Validation:', val)

## 4. Đọc biểu đồ quá trình học

Biểu đồ có hai ô: **Accuracy** (trái) và **Loss** (phải). Mỗi ô có đường *train* và đường *validation*.

| Hiện tượng | Ý nghĩa |
|---|---|
| Cả hai đường accuracy cùng tăng, loss cùng giảm, sát nhau | Model học tốt |
| Train accuracy rất cao nhưng validation thấp hơn nhiều, val loss tăng trở lại | **Overfitting**: model học vẹt tập train, kém với ảnh mới |
| Cả hai đều thấp | **Underfitting**: model quá đơn giản hoặc train chưa đủ |
| Đường validation dao động mạnh | Tập validation nhỏ hoặc learning rate hơi lớn |

In [ ]:
hist_path = plot_history(history, os.path.join(REPORTS_DIR, 'baseline_training_history.png'))
Image(hist_path)

## 5. Đánh giá trên tập test

Bước này chấm điểm model trên tập **test**, tức ảnh model chưa hề thấy khi học. In ra và lưu 3 thứ:

- **Test accuracy / loss:** độ chính xác và độ sai lệch tổng thể.
- **Confusion matrix (ma trận nhầm lẫn):** bảng 2×2. Hàng là nhãn thật, cột là nhãn model đoán. Ô chéo chính (trái trên, phải dưới) là số ảnh đoán đúng. Hai ô còn lại là số ảnh đoán sai: ảnh `fresh` bị đoán thành `rotten` và ngược lại.
- **Classification report:** gồm *precision* (trong số ảnh model gọi là `rotten`, bao nhiêu % đúng), *recall* (trong số ảnh thật sự `rotten`, model tìm ra bao nhiêu %) và *f1-score* (trung bình của hai chỉ số trên).

**Lưu ý quan trọng:** tập test có cùng kiểu ảnh với tập train, nên điểm test cao chưa chắc model tốt với ảnh ngoài đời (tay cầm, nền lộn xộn, nhiều quả). Hãy thử thêm ảnh thật qua API để kiểm tra.

In [ ]:
metrics = evaluate_model(model, test_gen, prefix='baseline_')
print(metrics)
Image(os.path.join(REPORTS_DIR, 'baseline_confusion_matrix.png'))

## Bước tiếp theo

Ghi lại `val_acc` và `test_acc` của baseline làm mốc. Sang notebook **03** để thử thêm các cấu hình khác và xem có cấu hình nào vượt baseline không.